In [ ]:
from pathlib import Path
from strategies import SPLIT_STRATEGIES
from dataset_factory import build_split_dataset_with_sets, get_output_dir_for_split
from persistence import persist_patient_bundles

## Build Graphs and Load Resources

Load the FHIR resources from compressed NDJSON files and build forward/reverse reference graphs.

In [ ]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

In [ ]:
# Select strategy and dataset name
strategy = SPLIT_STRATEGIES["random"]
dataset_name = "random"
seed = 42  # Set seed for reproducibility

print(f"Strategy: {strategy.__name__}")
print(f"Dataset name: {dataset_name}")
print(f"Random seed: {seed}")


In [ ]:
print("Building split dataset...")
results = build_split_dataset_with_sets(
    strategy=strategy,
    resources_map=resources_map,
    reverse_graph=reverse_graph,
    forward_graph=forward_graph,
    seed=seed
)

print(f"Built datasets for {len(results)} splits")
for split_name, locations in results.items():
    print(f"  {split_name}: {sum(len(bundles) for bundles in locations.values())} patients")

## Persist Datasets to Disk

Save the split bundles to `input/{dataset_name}/hospital_A` and `input/{dataset_name}/hospital_B`.

In [ ]:
print("Persisting bundles...")

total_persisted = 0
for split_name, locations in results.items():
    for location_name, patient_bundles in locations.items():
        if not patient_bundles:  # Skip empty locations
            continue
        
        output_dir = get_output_dir_for_split(
            base_dir=f"input/{dataset_name}",
            split_set=split_name,
            location_name=location_name
        )
        
        persisted = persist_patient_bundles(
            patient_bundles,
            output_dir,
            resources_map
        )
        
        print(f"Saved {persisted} bundles to {output_dir}")
        total_persisted += persisted

print(f"\nTotal bundles persisted: {total_persisted}")